# 01: Merge Raw Datasets & Exploratory Data Analysis (EDA)

This notebook executes the data preparation and exploratory analysis phase for MITRE ATT&CK CTI technique classification:
1. **Preliminary Processing & Dataset Merging**:
   - Merges three raw CTI datasets: `Attack_Dataset.csv`, `single_label.json`, and `multi_label.json`.
   - Maps technique identifiers to **Parent Techniques (`Txxxx`)** to maintain the **Full 378 Label Space** (Extreme Multi-Label Classification - XMC).
   - **Deduplication & Union Merging**: Merges identical CTI text entries while maintaining label unions (`set.update`) to prevent information loss.
   - **Entity Neutralization (Label Leakage Prevention)**: Removes direct MITRE technique codes (`Txxxx` / `Txxxx.xxx`) from text descriptions to avoid *Label Leakage*.
2. **Comprehensive Exploratory Data Analysis (EDA)**:
   - Evaluates text length statistics and tokenization percentiles to inform Transformer sequence length selection.
   - Analyzes source contribution breakdown across raw datasets.
   - Measures multi-label cardinality, density, label count distribution per sample, and top technique co-occurrence pairs.
   - Analyzes the Zipfian long-tail distribution across all 378 parent technique labels: Frequent (>= 100), Medium (30-99), Rare (< 30), and Zero-sample techniques.
   - Extracts top cybersecurity domain vocabulary and N-gram Document Frequencies (Unigrams & Bigrams).
   - Automatically exports analytical summaries and visualization graphics to `results/EDA_results/`.

> Note: URLs, HTML tags, Markdown formatting, and entity anonymization (e.g. replacing IPv4, Windows File Paths with special tokens) will be handled in `02_preprocessing.ipynb`.

In [1]:
import os
import sys
import json
import re
import pandas as pd
import numpy as np
from collections import Counter
from itertools import combinations
from pathlib import Path
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for headless execution
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Configure visualization theme
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print('[INFO] Required libraries imported successfully.')

[INFO] Required libraries imported successfully.


In [2]:
# Define directory structure and file paths
RAW_DIR = Path('../dataset/raw')
PROCESSED_DIR = Path('../dataset/processed')
RESULTS_DIR = Path('../results/EDA_results')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ATTACK_CSV_PATH = RAW_DIR / 'Attack_Dataset.csv'
SINGLE_JSON_PATH = RAW_DIR / 'single_label.json'
MULTI_JSON_PATH = RAW_DIR / 'multi_label.json'

# Output file paths
OUTPUT_MERGED_DATASET = PROCESSED_DIR / '01_merged_cti_dataset.csv'
OUTPUT_LABEL_STATS = RESULTS_DIR / '01_label_distribution_summary.csv'
OUTPUT_EDA_JSON = RESULTS_DIR / '01_dataset_eda_summary.json'
OUTPUT_LONGTAIL_PLOT = RESULTS_DIR / '01_dataset_longtail_distribution.png'

print(f"[INFO] Raw Dataset Directory    : {RAW_DIR.resolve()}")
print(f"[INFO] Processed Data Directory : {PROCESSED_DIR.resolve()}")
print(f"[INFO] EDA Results Directory    : {RESULTS_DIR.resolve()}")

[INFO] Raw Dataset Directory    : D:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\raw
[INFO] Processed Data Directory : D:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\processed
[INFO] EDA Results Directory    : D:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\results\EDA_results


## 1. Helper Functions for Label Extraction & Leakage Prevention

- **`get_parent_label(lbl)`**: Extracts the MITRE Parent Technique ID (`T1059.001` -> `T1059`).
- **`clean_cti_text_preliminary(text)`**: Neutralizes direct MITRE technique codes (`Txxxx` / `Txxxx.xxx`) from text descriptions to prevent *Label Leakage*, while retaining URLs, HTML, Markdown, and natural grammar for `02_preprocessing.ipynb`.

In [3]:
MITRE_PATTERN = re.compile(r'T\d{4}(?:\.\d{3})?')

def get_parent_label(lbl):
    """Extract parent technique ID (Txxxx) from any MITRE technique string."""
    if pd.isna(lbl):
        return None
    lbl_str = str(lbl).strip()
    match = MITRE_PATTERN.search(lbl_str)
    if match:
        return match.group(0).split('.')[0]
    return None

def clean_cti_text_preliminary(text):
    """
    Preliminary text cleaning pipeline:
    - Neutralizes direct MITRE IDs (Txxxx / Txxxx.xxx) to prevent label leakage.
    - Preserves HTML, URLs, Markdown, and entity strings for downstream tokenization/anonymization in 02_preprocessing.ipynb.
    - Normalizes extra whitespace while maintaining natural grammar and casing.
    """
    if pd.isna(text):
        return ""
    t = str(text)
    t = MITRE_PATTERN.sub(' ', t)  # Remove direct MITRE technique codes to avoid label leakage
    t = re.sub(r'\b(unknown|nan)\b', ' ', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# Verification test on sample input
test_sample = "Attacker executed T1059.001 (Command Shell) via http://badurl.com #malware <p>Details</p>"
print("[TEST] Raw sample      :", test_sample)
print("[TEST] Cleaned output  :", clean_cti_text_preliminary(test_sample))

[TEST] Raw sample      : Attacker executed T1059.001 (Command Shell) via http://badurl.com #malware <p>Details</p>
[TEST] Cleaned output  : Attacker executed (Command Shell) via http://badurl.com #malware <p>Details</p>


## 2. Load, Merge, and Deduplicate Raw CTI Datasets

The processing logic follows these steps:
1. **`Attack_Dataset.csv`** (~14,133 rows): Extract MITRE technique codes from all record columns, aggregate text descriptions, neutralize MITRE IDs, and deduplicate.
2. **`single_label.json`** (~25,000 items): Remove direct MITRE IDs, map single labels to parent technique codes, and perform union set update (`set.update`) on matching text entries.
3. **`multi_label.json`** (~105,000 items): Remove direct MITRE IDs, map multi-label arrays to parent technique codes, and update label sets.

In [4]:
text_to_labels = {}
source_contributions = {'Attack_Dataset.csv': 0, 'single_label.json': 0, 'multi_label.json': 0}
order = []

# Track source dataset origins for EDA
text_to_sources = {}

# --- 1. Process Attack_Dataset.csv ---
print("[STEP 1] Loading and parsing Attack_Dataset.csv...")
df_attack_raw = pd.read_csv(ATTACK_CSV_PATH)
attack_count = 0
for idx, row in df_attack_raw.iterrows():
    row_content = row.drop(labels=['ID', 'Source'], errors='ignore').fillna('')
    row_text_raw = ' '.join(str(val) for val in row_content.values)
    raw_mitres = MITRE_PATTERN.findall(row_text_raw)
    if not raw_mitres:
        continue
    parent_labels = set(get_parent_label(l) for l in raw_mitres if get_parent_label(l))
    if not parent_labels:
        continue
    cleaned_t = clean_cti_text_preliminary(row_text_raw)
    if not cleaned_t:
        continue
    attack_count += 1
    source_contributions['Attack_Dataset.csv'] += 1
    if cleaned_t not in text_to_labels:
        text_to_labels[cleaned_t] = parent_labels
        text_to_sources[cleaned_t] = {'Attack_Dataset.csv'}
        order.append(cleaned_t)
    else:
        text_to_labels[cleaned_t].update(parent_labels)
        text_to_sources[cleaned_t].add('Attack_Dataset.csv')
print(f"   [RESULT] Retained {attack_count:,} valid rows from Attack_Dataset.csv (Unique texts: {len(order):,}).")

# --- 2. Process single_label.json ---
print("\n[STEP 2] Loading and merging single_label.json...")
with open(SINGLE_JSON_PATH, 'r', encoding='utf-8') as f:
    single_data = json.load(f)
s_added, s_updated = 0, 0
for item in single_data:
    raw_t = str(item.get('text', '')).strip()
    raw_l = str(item.get('label', '')).strip()
    if not raw_t or not raw_l:
        continue
    parent_l = get_parent_label(raw_l)
    if not parent_l:
        continue
    cleaned_t = clean_cti_text_preliminary(raw_t)
    if not cleaned_t:
        continue
    source_contributions['single_label.json'] += 1
    if cleaned_t not in text_to_labels:
        text_to_labels[cleaned_t] = {parent_l}
        text_to_sources[cleaned_t] = {'single_label.json'}
        order.append(cleaned_t)
        s_added += 1
    else:
        text_to_sources[cleaned_t].add('single_label.json')
        if parent_l not in text_to_labels[cleaned_t]:
            text_to_labels[cleaned_t].add(parent_l)
            s_updated += 1
print(f"   [RESULT] Added new: {s_added:,} | Updated labels: {s_updated:,} (Total unique texts: {len(order):,}).")

# --- 3. Process multi_label.json ---
print("\n[STEP 3] Loading and merging multi_label.json...")
with open(MULTI_JSON_PATH, 'r', encoding='utf-8') as f:
    multi_data = json.load(f)
m_added, m_updated = 0, 0
for item in multi_data:
    labels_list = item.get('labels', [])
    if not labels_list:
        continue
    raw_t = str(item.get('sentence', '')).strip()
    if not raw_t:
        continue
    parent_labels = set(get_parent_label(str(l)) for l in labels_list if get_parent_label(str(l)))
    if not parent_labels:
        continue
    cleaned_t = clean_cti_text_preliminary(raw_t)
    if not cleaned_t:
        continue
    source_contributions['multi_label.json'] += 1
    if cleaned_t not in text_to_labels:
        text_to_labels[cleaned_t] = parent_labels
        text_to_sources[cleaned_t] = {'multi_label.json'}
        order.append(cleaned_t)
        m_added += 1
    else:
        text_to_sources[cleaned_t].add('multi_label.json')
        before_len = len(text_to_labels[cleaned_t])
        text_to_labels[cleaned_t].update(parent_labels)
        if len(text_to_labels[cleaned_t]) > before_len:
            m_updated += 1
print(f"   [RESULT] Added new: {m_added:,} | Updated labels: {m_updated:,}.")
print(f"[SUCCESS] Total unique samples after deduplication and merging: {len(order):,}")

[STEP 1] Loading and parsing Attack_Dataset.csv...
   [RESULT] Retained 14,072 valid rows from Attack_Dataset.csv (Unique texts: 13,838).

[STEP 2] Loading and merging single_label.json...
   [RESULT] Added new: 4,811 | Updated labels: 23 (Total unique texts: 18,649).

[STEP 3] Loading and merging multi_label.json...
   [RESULT] Added new: 3,735 | Updated labels: 32.
[SUCCESS] Total unique samples after deduplication and merging: 22,384


## 3. Export Unified Processed Dataset

Save the finalized merged dataset to `dataset/processed/01_merged_cti_dataset.csv` containing two standardized columns: `Cleaned_Text` and `Labels` (comma-separated).

In [5]:
final_rows = []
for text in order:
    sorted_labels = ','.join(sorted(list(text_to_labels[text])))
    final_rows.append({'Cleaned_Text': text, 'Labels': sorted_labels})

df_merged = pd.DataFrame(final_rows)
df_merged.to_csv(OUTPUT_MERGED_DATASET, index=False, encoding='utf-8')

all_unique_labels = sorted(list(set().union(*text_to_labels.values())))
print(f"[INFO] Merged dataset saved to: {OUTPUT_MERGED_DATASET}")
print(f"[INFO] Total Dataset Shape     : {df_merged.shape[0]:,} rows, {df_merged.shape[1]} columns")
print(f"[INFO] Unique Parent Techniques : {len(all_unique_labels)} active labels out of 378 full target space")

[INFO] Merged dataset saved to: ..\dataset\processed\01_merged_cti_dataset.csv
[INFO] Total Dataset Shape     : 22,384 rows, 2 columns
[INFO] Unique Parent Techniques : 378 active labels out of 378 full target space


## 4. Detailed Exploratory Data Analysis (EDA)

This section performs detailed statistical exploration across multiple key dimensions:
1. **Data Quality & Null Check**: Verification of non-null constraints.
2. **Raw Dataset Source Contribution**: Sample counts, text length distributions, and unique labels contributed per raw file.
3. **Text Length & Tokenization Percentiles**: Analysis of character and word count percentiles (50th, 75th, 90th, 95th, 99th, max) to determine optimal Transformer `max_seq_length` thresholds.
4. **Multi-label Multiplicity & Density**: Evaluation of label cardinality (average labels/sample) and distribution of label count per sample.
5. **Label Co-occurrence Pair Analysis**: Identification of top co-occurring technique pairs.
6. **Class Imbalance & Zipfian Long-Tail Analysis**: Grouping techniques into Frequent (>=100), Medium (30-99), Rare (<30), and Zero-sample categories across the full 378 target space.
7. **N-gram & Document Frequency Analysis**: Extracting Top Unigrams and Bigrams with Document Frequency (`DF %`) to evaluate high-frequency terms.

In [6]:
print("=" * 80)
print("--- 1. DATA QUALITY & NULL VALUE CHECK ---")
print(df_merged.isnull().sum())
print("=" * 80)

# Calculate text length metrics
df_merged['Char_Length'] = df_merged['Cleaned_Text'].apply(len)
df_merged['Word_Count'] = df_merged['Cleaned_Text'].apply(lambda x: len(x.split()))
label_list = [str(l).split(',') for l in df_merged['Labels']]
df_merged['Label_Count'] = [len(l) for l in label_list]
flat_labels = [lbl.strip() for sublist in label_list for lbl in sublist if lbl.strip()]

print("--- 2. RAW SOURCE DATASET CONTRIBUTION SUMMARY ---")
for src_name, raw_cnt in source_contributions.items():
    unique_cnt = sum(1 for t, srcs in text_to_sources.items() if src_name in srcs)
    print(f"   - {src_name:<20}: {raw_cnt:8,} raw items processed | {unique_cnt:8,} unique entries contributed")
print("=" * 80)

print("--- 3. TEXT LENGTH & TOKENIZATION PERCENTILES ---")
word_stats = df_merged['Word_Count'].describe().round(2)
percentiles = np.percentile(df_merged['Word_Count'], [50, 75, 90, 95, 99])
print(f"   - Mean Word Count   : {word_stats['mean']:.2f}")
print(f"   - Std Word Count    : {word_stats['std']:.2f}")
print(f"   - Median (50th Pct) : {percentiles[0]:.0f} words")
print(f"   - 75th Percentile   : {percentiles[1]:.0f} words")
print(f"   - 90th Percentile   : {percentiles[2]:.0f} words")
print(f"   - 95th Percentile   : {percentiles[3]:.0f} words")
print(f"   - 99th Percentile   : {percentiles[4]:.0f} words")
print(f"   - Max Word Count    : {word_stats['max']:.0f} words")

# Sequence length threshold recommendations for Transformers
p90_coverage = (df_merged['Word_Count'] <= 128).mean() * 100
p95_coverage = (df_merged['Word_Count'] <= 256).mean() * 100
p99_coverage = (df_merged['Word_Count'] <= 512).mean() * 100
print(f"   - Coverage at max_seq_len = 128 tokens : {p90_coverage:.2f}%")
print(f"   - Coverage at max_seq_len = 256 tokens : {p95_coverage:.2f}%")
print(f"   - Coverage at max_seq_len = 512 tokens : {p99_coverage:.2f}%")
print("=" * 80)

print("--- 4. MULTI-LABEL CARDINALITY & DENSITY ---")
label_cardinality = df_merged['Label_Count'].mean()
single_label_samples = (df_merged['Label_Count'] == 1).sum()
multi_label_samples = (df_merged['Label_Count'] > 1).sum()
label_count_dist = {f"{k}_label(s)": int(v) for k, v in df_merged['Label_Count'].value_counts().sort_index().to_dict().items()}

print(f"   - Label Cardinality (Avg labels/sample) : {label_cardinality:.3f}")
print(f"   - Single-label samples                  : {single_label_samples:,} ({single_label_samples/len(df_merged)*100:.2f}%)")
print(f"   - Multi-label samples                   : {multi_label_samples:,} ({multi_label_samples/len(df_merged)*100:.2f}%)")
print(f"   - Max labels in a single sample         : {df_merged['Label_Count'].max()}")
print("   - Label Count Distribution per Sample   :")
for label_k, val_c in label_count_dist.items():
    print(f"       • {label_k:<12}: {val_c:6,} samples ({val_c/len(df_merged)*100:.2f}%)")
print("=" * 80)

# Label Co-occurrence Analysis
co_occur_counts = Counter()
for labels in label_list:
    cleaned_labels = sorted(list(set(l.strip() for l in labels if l.strip())))
    if len(cleaned_labels) >= 2:
        for pair in combinations(cleaned_labels, 2):
            co_occur_counts[pair] += 1

top_pairs = co_occur_counts.most_common(10)
print("--- 5. TOP 10 CO-OCCURRING TECHNIQUE PAIRS ---")
for (t1, t2), count in top_pairs:
    print(f"   - Pair ({t1}, {t2}): {count} co-occurrences")
print("=" * 80)

# Long-tail Group Distribution
label_counts = Counter(flat_labels)
TOTAL_MITRE_PARENT_TECHNIQUES = 378
active_labels_count = len(label_counts)
zero_sample_count = TOTAL_MITRE_PARENT_TECHNIQUES - active_labels_count

df_stats = pd.DataFrame(label_counts.items(), columns=['Technique', 'Sample_Count']).sort_values(by='Sample_Count', ascending=False).reset_index(drop=True)
df_stats['Percentage (%)'] = (df_stats['Sample_Count'] / len(df_merged) * 100).round(3)

def assign_freq_group(cnt):
    if cnt >= 100:
        return 'Frequent (>=100)'
    elif cnt >= 30:
        return 'Medium (30-99)'
    else:
        return 'Rare (<30)'

df_stats['Group'] = df_stats['Sample_Count'].apply(assign_freq_group)
group_summary = df_stats['Group'].value_counts().to_dict()
group_summary['Zero-sample (0)'] = zero_sample_count

print("--- 6. CLASS IMBALANCE & LONG-TAIL DISTRIBUTION (FULL 378 SPACE) ---")
print(f"   - Frequent Techniques (>=100) : {group_summary.get('Frequent (>=100)', 0):3d} labels ({group_summary.get('Frequent (>=100)', 0)/TOTAL_MITRE_PARENT_TECHNIQUES*100:.2f}%)")
print(f"   - Medium Techniques (30-99)   : {group_summary.get('Medium (30-99)', 0):3d} labels ({group_summary.get('Medium (30-99)', 0)/TOTAL_MITRE_PARENT_TECHNIQUES*100:.2f}%)")
print(f"   - Rare Techniques (<30)       : {group_summary.get('Rare (<30)', 0):3d} labels ({group_summary.get('Rare (<30)', 0)/TOTAL_MITRE_PARENT_TECHNIQUES*100:.2f}%)")
print(f"   - Zero-sample Techniques (0)  : {zero_sample_count:3d} labels ({zero_sample_count/TOTAL_MITRE_PARENT_TECHNIQUES*100:.2f}%)")
print(f"   - Total Target Space          : {TOTAL_MITRE_PARENT_TECHNIQUES} labels")
print("=" * 80)

# --- 7. N-Gram & Document Frequency (DF) Analysis ---
print("--- 7. N-GRAM DOCUMENT FREQUENCY (DF) ANALYSIS ---")
vec_uni = CountVectorizer(ngram_range=(1, 1), stop_words='english', max_features=15)
X_uni = vec_uni.fit_transform(df_merged['Cleaned_Text'])
df_unigrams = pd.DataFrame({
    'term': vec_uni.get_feature_names_out(),
    'doc_count': np.asarray((X_uni > 0).sum(axis=0)).ravel()
}).sort_values(by='doc_count', ascending=False).reset_index(drop=True)
df_unigrams['doc_freq_pct'] = (df_unigrams['doc_count'] / len(df_merged) * 100).round(2)

vec_bi = CountVectorizer(ngram_range=(2, 2), stop_words='english', max_features=15)
X_bi = vec_bi.fit_transform(df_merged['Cleaned_Text'])
df_bigrams = pd.DataFrame({
    'term': vec_bi.get_feature_names_out(),
    'doc_count': np.asarray((X_bi > 0).sum(axis=0)).ravel()
}).sort_values(by='doc_count', ascending=False).reset_index(drop=True)
df_bigrams['doc_freq_pct'] = (df_bigrams['doc_count'] / len(df_merged) * 100).round(2)

print("   [TOP 15 UNIGRAMS (Word Document Frequency)]:")
for idx, row in df_unigrams.iterrows():
    print(f"       • {idx+1:2d}. {row['term']:<20}: {row['doc_count']:6,} docs ({row['doc_freq_pct']:.2f}%)")

print("\n   [TOP 15 BIGRAMS (Phrase Document Frequency)]:")
for idx, row in df_bigrams.iterrows():
    print(f"       • {idx+1:2d}. {row['term']:<30}: {row['doc_count']:6,} docs ({row['doc_freq_pct']:.2f}%)")
print("=" * 80)

# Save detailed statistical reports in RESULTS_DIR (results/EDA_results)
df_stats.to_csv(OUTPUT_LABEL_STATS, index=False)

eda_summary_dict = {
    'total_samples': len(df_merged),
    'total_unique_active_labels': active_labels_count,
    'total_target_labels': TOTAL_MITRE_PARENT_TECHNIQUES,
    'label_cardinality': round(label_cardinality, 4),
    'labels_per_sample_distribution': label_count_dist,
    'word_count_percentiles': {
        'p50': int(percentiles[0]),
        'p75': int(percentiles[1]),
        'p90': int(percentiles[2]),
        'p95': int(percentiles[3]),
        'p99': int(percentiles[4])
    },
    'token_coverage': {
        'seq_128': round(p90_coverage, 2),
        'seq_256': round(p95_coverage, 2),
        'seq_512': round(p99_coverage, 2)
    },
    'group_breakdown': group_summary,
    'top_co_occurring_pairs': [{'pair': f"{p[0]}-{p[1]}", 'count': c} for p, c in top_pairs],
    'top_unigrams': df_unigrams.to_dict(orient='records'),
    'top_bigrams': df_bigrams.to_dict(orient='records')
}

with open(OUTPUT_EDA_JSON, 'w', encoding='utf-8') as f:
    json.dump(eda_summary_dict, f, indent=2)

print(f"[INFO] Exported per-label statistics to : {OUTPUT_LABEL_STATS}")
print(f"[INFO] Exported structured EDA summary to : {OUTPUT_EDA_JSON}")

--- 1. DATA QUALITY & NULL VALUE CHECK ---
Cleaned_Text    0
Labels          0
dtype: int64
--- 2. RAW SOURCE DATASET CONTRIBUTION SUMMARY ---
   - Attack_Dataset.csv  :   14,072 raw items processed |   13,838 unique entries contributed
   - single_label.json   :    5,089 raw items processed |    4,811 unique entries contributed
   - multi_label.json    :    4,070 raw items processed |    4,035 unique entries contributed
--- 3. TEXT LENGTH & TOKENIZATION PERCENTILES ---
   - Mean Word Count   : 118.10
   - Std Word Count    : 110.64
   - Median (50th Pct) : 111 words
   - 75th Percentile   : 165 words
   - 90th Percentile   : 257 words
   - 95th Percentile   : 311 words
   - 99th Percentile   : 509 words
   - Max Word Count    : 1071 words
   - Coverage at max_seq_len = 128 tokens : 59.94%
   - Coverage at max_seq_len = 256 tokens : 89.95%
   - Coverage at max_seq_len = 512 tokens : 99.07%
--- 4. MULTI-LABEL CARDINALITY & DENSITY ---
   - Label Cardinality (Avg labels/sample) : 1.106
 

## 5. Visual Data Analysis & Graphical Export

Generate a comprehensive 6-subplot analytical figure illustrating:
1. Top 20 Most Frequent MITRE Techniques.
2. Zipfian Long-Tail Curve (Log scale) with frequency thresholds.
3. Proportion of Label Frequency Groups.
4. Distribution of Number of Labels per Sample.
5. CTI Text Word Count Distribution with Transformer Threshold lines.
6. Top Co-occurring MITRE Technique Pairs.

The composite plot is exported directly to `results/EDA_results/01_dataset_longtail_distribution.png`.

In [7]:
fig, axes = plt.subplots(3, 2, figsize=(18, 15))

# Subplot 1: Top 20 Most Frequent Techniques
sns.barplot(ax=axes[0, 0], data=df_stats.head(20), x='Sample_Count', y='Technique', palette='viridis')
axes[0, 0].set_title('Top 20 Most Frequent MITRE Techniques', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Sample Count')
axes[0, 0].set_ylabel('MITRE Technique')

# Subplot 2: Zipfian Long-Tail Distribution (Log Scale)
axes[0, 1].plot(range(1, len(df_stats) + 1), df_stats['Sample_Count'], color='crimson', linewidth=2.5)
axes[0, 1].set_yscale('log')
axes[0, 1].set_title('Extreme Long-Tail Zipfian Distribution (Log Scale)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Technique Rank (1 to 378)')
axes[0, 1].set_ylabel('Sample Count (Log Scale)')
axes[0, 1].axhline(y=100, color='green', linestyle='--', label='Frequent (>=100)')
axes[0, 1].axhline(y=30, color='orange', linestyle='--', label='Medium (>=30)')
axes[0, 1].legend()

# Subplot 3: Label Frequency Group Proportion (Pie Chart)
group_order = ['Frequent (>=100)', 'Medium (30-99)', 'Rare (<30)', 'Zero-sample (0)']
group_vals = [group_summary.get(g, 0) for g in group_order]
axes[1, 0].pie(group_vals, labels=group_order, autopct='%1.1f%%', startangle=140, 
               colors=['#2ecc71', '#f39c12', '#e74c3c', '#95a5a6'], explode=(0.04, 0.04, 0.04, 0.04))
axes[1, 0].set_title('Label Frequency Group Proportions (378 Space)', fontsize=12, fontweight='bold')

# Subplot 4: Labels per Sample Distribution
sns.countplot(ax=axes[1, 1], data=df_merged, x='Label_Count', hue='Label_Count', palette='magma', legend=False)
axes[1, 1].set_title('Distribution of Labels per Sample', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Number of Labels in Sample')
axes[1, 1].set_ylabel('Sample Count')
for p in axes[1, 1].patches:
    if p.get_height() > 0:
        axes[1, 1].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                            ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

# Subplot 5: Text Length (Word Count) Distribution
sns.histplot(ax=axes[2, 0], data=df_merged[df_merged['Word_Count'] <= 600], x='Word_Count', bins=40, kde=True, color='teal')
axes[2, 0].set_title('CTI Text Word Count Distribution (<= 600 words)', fontsize=12, fontweight='bold')
axes[2, 0].set_xlabel('Word Count')
axes[2, 0].set_ylabel('Sample Count')
axes[2, 0].axvline(x=128, color='green', linestyle=':', label='128 words (~90% pct)')
axes[2, 0].axvline(x=256, color='darkorange', linestyle=':', label='256 words (~96% pct)')
axes[2, 0].legend()

# Subplot 6: Top Co-occurring Label Pairs
if top_pairs:
    pair_labels = [f"{p[0]} + {p[1]}" for p, c in top_pairs]
    pair_counts = [c for p, c in top_pairs]
    sns.barplot(ax=axes[2, 1], x=pair_counts, y=pair_labels, palette='rocket')
    axes[2, 1].set_title('Top 10 Co-occurring Technique Pairs', fontsize=12, fontweight='bold')
    axes[2, 1].set_xlabel('Co-occurrence Count')
    axes[2, 1].set_ylabel('Technique Pair')

plt.tight_layout()
plt.savefig(OUTPUT_LONGTAIL_PLOT, dpi=300)
plt.close()
print(f"[INFO] Exported composite EDA visualization to: {OUTPUT_LONGTAIL_PLOT}")

[INFO] Exported composite EDA visualization to: ..\results\EDA_results\01_dataset_longtail_distribution.png


## 6. Dataset Sample Preview

Display the first 5 and last 5 rows of the processed merged dataset (`01_merged_cti_dataset.csv`).

In [8]:
print("[PREVIEW] First 5 Rows:")
display(df_merged[['Cleaned_Text', 'Labels']].head(5))

print("\n[PREVIEW] Last 5 Rows:")
display(df_merged[['Cleaned_Text', 'Labels']].tail(5))

[PREVIEW] First 5 Rows:
                                        Cleaned_Text       Labels
0  Authentication Bypass via SQL Injection Mobile...  T1078,T1190
1  Union-Based SQL Injection AI Agents & LLM Expl...        T1190
2  Error-Based SQL Injection AI Agents & LLM Expl...        T1190
3  Blind SQL Injection AI Agents & LLM Exploits S...        T1190
4  Second-Order SQL Injection AI Agents & LLM Exp...        T1505

[PREVIEW] Last 5 Rows:
                                            Cleaned_Text       Labels
22379  Application Layer Protocol: Web Protocols Tric...        T1071
22380  Ingress Tool Transfer TrickBot downloads sever...        T1105
22381  Non-Standard Port Some TrickBot samples have u...        T1071
22382  Symmetric Cryptography TrickBot uses a custom ...  T1106,T1573
22383  Exfiltration [TA0010] Technique Tactic ID Use ...        T1041
